In [ ]:
import os
import sys
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import confusion_matrix, f1_score, classification_report, roc_auc_score, average_precision_score
import numpy as np
PROJECT_ROOT = os.path.abspath("..")    
sys.path.append(PROJECT_ROOT)

In [ ]:

from dataset.otto_final import TraceOttoDataset
from utils.SplitData import split_data_Train_Val_Test
from utils.feature_engineering import get_between_features, get_elapsed_feature
from utils.plot_confussion_matrix import plot_confusion_matrix
from utils.training_utils import initialize_TRACE_model, append_probs_and_true, concate_probs_true, search_best_f1_thr
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [ ]:
dataset_processed  = TraceOttoDataset(
    file_name='../train.jsonl',
    input_seq_len=64,
    min_timestamps_per_sample=32,
)
train_loader, validation_loader, test_loader = split_data_Train_Val_Test(dataset_processed, batch_size=128)


print(len(train_loader.dataset))
print(len(validation_loader.dataset))
print(len(test_loader.dataset))


In [ ]:
trace_model = initialize_TRACE_model(dataset_processed, num_classes=3, device=device)

#Need to be changed after the training, to have the model that was trained on
final_model_load = torch.load("Model_TRACE_MLT_ATC_SAT_MLP_FinalVersion_70_80_.pt", weights_only=False, map_location=device)

trace_model.load_state_dict(final_model_load["model_state_dict"])
best_thr_ATC = final_model_load["best_global_threshold"]["ATC"]
best_thr_SAT = final_model_load["best_global_threshold"]["SAT"]
best_thr_MAP = final_model_load["best_global_threshold"]["MAP"]

trace_model.eval()
print(
    f"Loaded model | ATC: {best_thr_ATC:.4f} | "
    f"SAT: {best_thr_SAT:.4f} | "
    f"MAP: {best_thr_MAP:.4f}"
)


In [ ]:
test_correct_ATC = 0
test_total_ATC = 0
test_correct_SAT = 0
test_total_SAT = 0
test_correct_MAP = 0
test_total_MAP = 0

y_true_ATC = []
y_pred_ATC = []
y_prob_ATC = []

y_true_SAT = []
y_pred_SAT = []
y_prob_SAT = []

y_true_MAP = []
y_pred_MAP = []
y_prob_MAP = []
trace_model.eval()
criterion = torch.nn.BCEWithLogitsLoss()

test_loss = 0.0

y_true_ATC, y_prob_ATC = [], []
y_true_SAT, y_prob_SAT = [], []
y_true_MAP, y_prob_MAP = [], []

with torch.no_grad():
    for inputs_test, targets_test in test_loader:
        t_ATC = targets_test["ATC"].unsqueeze(1).to(device).float()
        t_SAT = targets_test["SAT"].unsqueeze(1).to(device).float()
        t_MAP = targets_test["MAP"].unsqueeze(1).to(device).float()

        inputs_test = {k: v.to(device) for k, v in inputs_test.items()}
        delta_elapsed = get_elapsed_feature(inputs_test["timestamps"]).to(device).float()
        delta_between = get_between_features(inputs_test["timestamps"]).to(device).float()

        logits = trace_model(inputs_test["aid"], inputs_test["type"], delta_elapsed, delta_between)

        loss_ATC = criterion(logits[:, 0:1], t_ATC)
        loss_SAT = criterion(logits[:, 1:2], t_SAT)
        loss_MAP = criterion(logits[:, 2:3], t_MAP)

        loss_batch = (loss_ATC + loss_SAT + loss_MAP) / 3
        test_loss += loss_batch.item()

        append_probs_and_true(logits[:, 0:1], t_ATC, y_prob_ATC, y_true_ATC)
        append_probs_and_true(logits[:, 1:2], t_SAT, y_prob_SAT, y_true_SAT)
        append_probs_and_true(logits[:, 2:3], t_MAP, y_prob_MAP, y_true_MAP)

test_loss /= len(test_loader)

probs_ATC, true_ATC = concate_probs_true(y_prob_ATC, y_true_ATC)
probs_SAT, true_SAT = concate_probs_true(y_prob_SAT, y_true_SAT)
probs_MAP, true_MAP = concate_probs_true(y_prob_MAP, y_true_MAP)


pred_ATC = (probs_ATC >= best_thr_ATC).astype(int)
pred_SAT = (probs_SAT >= best_thr_SAT).astype(int)
pred_MAP = (probs_MAP >= best_thr_MAP).astype(int)


f1_ATC = f1_score(true_ATC, pred_ATC, zero_division=0)
f1_SAT = f1_score(true_SAT, pred_SAT, zero_division=0)
f1_MAP = f1_score(true_MAP, pred_MAP, zero_division=0)

macro_f1_ATC = f1_score(true_ATC, pred_ATC, average="macro", zero_division=0)
macro_f1_SAT = f1_score(true_SAT, pred_SAT, average="macro", zero_division=0)
macro_f1_MAP = f1_score(true_MAP, pred_MAP, average="macro", zero_division=0)

auroc_ATC = roc_auc_score(true_ATC, probs_ATC) 
auroc_SAT = roc_auc_score(true_SAT, probs_SAT) 
auroc_MAP = roc_auc_score(true_MAP, probs_MAP) 

auprc_ATC = average_precision_score(true_ATC, probs_ATC)
auprc_SAT = average_precision_score(true_SAT, probs_SAT)
auprc_MAP = average_precision_score(true_MAP, probs_MAP)

cm_ATC = confusion_matrix(true_ATC, pred_ATC)
cm_SAT = confusion_matrix(true_SAT, pred_SAT)
cm_MAP = confusion_matrix(true_MAP, pred_MAP)



test_accuracy_ATC = (pred_ATC == true_ATC).mean()
test_accuracy_SAT = (pred_SAT == true_SAT).mean()
test_accuracy_MAP = (pred_MAP == true_MAP).mean()
test_accuracy_global = np.mean([test_accuracy_ATC, test_accuracy_SAT, test_accuracy_MAP])


f1_mean = (f1_ATC + f1_SAT + f1_MAP) / 3
macro_f1_mean = (macro_f1_ATC + macro_f1_SAT + macro_f1_MAP) / 3
auroc_mean = (auroc_ATC + auroc_SAT + auroc_MAP) / 3
auprc_mean = (auprc_ATC + auprc_SAT + auprc_MAP) / 3

print(f"Test accuracy global: {test_accuracy_global:.4f}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy mean: {np.mean([test_accuracy_ATC, test_accuracy_SAT, test_accuracy_MAP]):.4f}\n")

print(f"F1 Mean Score: {f1_mean:.4f}")
print(f"F1 ATC Score: {f1_ATC:.4f}")
print(f"F1 SAT Score: {f1_SAT:.4f}")
print(f"F1 MAP Score: {f1_MAP:.4f}\n")

print(f"Macro F1 ATC Score: {macro_f1_ATC:.4f}")
print(f"Macro F1 SAT Score: {macro_f1_SAT:.4f}")
print(f"Macro F1 MAP Score: {macro_f1_MAP:.4f}")
print(f"Macro F1 MEAN Score: {macro_f1_mean:.4f}\n")

print(f"AUROC ATC Score: {auroc_ATC:.4f}")
print(f"AUROC SAT Score: {auroc_SAT:.4f}")
print(f"AUROC MAP Score: {auroc_MAP:.4f}")
print(f"AUROC MEAN Score: {auroc_mean:.4f}\n")

print(f"AUPRC ATC Score: {auprc_ATC:.4f}")
print(f"AUPRC SAT Score: {auprc_SAT:.4f}")
print(f"AUPRC MAP Score: {auprc_MAP:.4f}")
print(f"AUPRC MEAN Score: {auprc_mean:.4f}\n")

print(f"Optimum Threshold ATC: {best_thr_ATC:.4f}")
print(f"Optimum Threshold SAT: {best_thr_SAT:.4f}")
print(f"Optimum Threshold MAP: {best_thr_MAP:.4f}")

print("\nClassification Report ATC:")
print(classification_report(true_ATC, pred_ATC, target_names=["ATC 0", "ATC 1"], zero_division=0))

print("\nClassification Report SAT:")
print(classification_report(true_SAT, pred_SAT, target_names=["SAT 0", "SAT 1"], zero_division=0))

print("\nClassification Report MAP:")
print(classification_report(true_MAP, pred_MAP, target_names=["MAP 0", "MAP 1"], zero_division=0))



print("Confusion Matrix: ATC\n", cm_ATC)
fig1 = plot_confusion_matrix(
    cm_ATC,
    name_task="ATC",
    name_classes=["class 0", "class 1"],
    save_path="confusion_ATC_MLT.pdf"
)
fig2 = plot_confusion_matrix(
    cm_SAT,
    name_task="SAT",
    name_classes=["class 0", "class 1"],
    save_path="confusion_SAT_MLT.pdf"
)
fig3 = plot_confusion_matrix(
    cm_MAP,
    name_task="MAP",
    name_classes=["class 0", "class 1"],
    save_path="confusion_MAP_MLT.pdf"
)
plt.show()
plt.close(fig1)
plt.close(fig2)
plt.close(fig3)
